In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import torch
import pandas as pd
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from datasets import Dataset
from utils.evaluation import *

In [3]:
df_train = pd.read_csv("lemmatized\\train_lemmatized.csv")
df_val = pd.read_csv("lemmatized\\validation_lemmatized.csv")
df_test = pd.read_csv("lemmatized\\test_lemmatized.csv")

df_expert_45 = pd.read_csv("lemmatized\\expert_core_45_lemmatized.csv")
df_expert_90 = pd.read_csv("lemmatized\\expert_core_90_lemmatized.csv")
df_expert_180 = pd.read_csv("lemmatized\\expert_core_180_lemmatized.csv")

## Preprocessing

In [4]:
def prepare_dataset(df, text_col='text_lemmatized', label_col='label'):
    df[text_col] = df[text_col].astype(str)

    mask = (df[text_col] != 'nan') & (df[text_col].str.strip() != '')
    df = df[mask].copy()

    dataset = Dataset.from_pandas(df[[text_col, label_col]])
    dataset = dataset.rename_column(text_col, "text")
    dataset = dataset.rename_column(label_col, "labels")
    return dataset

In [5]:
dataset_train = prepare_dataset(df_train)
dataset_val = prepare_dataset(df_val)
dataset_test = prepare_dataset(df_test)

dataset_expert_45 = prepare_dataset(df_expert_45)
dataset_expert_90 = prepare_dataset(df_expert_90)
dataset_expert_180 = prepare_dataset(df_expert_180)

In [6]:
model_name = "cointegrated/rubert-tiny2"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [7]:
def tokenize_function(examples):
    texts = examples["text"]
    
    cleaned_texts = []
    for t in texts:
        if t is None or (isinstance(t, float) and t != t):
            cleaned_texts.append("")
        elif not isinstance(t, str):
            cleaned_texts.append(str(t))
        else:
            cleaned_texts.append(t)
    
    return tokenizer(
        cleaned_texts, 
        truncation=True, 
        padding="max_length", 
        max_length=128
    )

In [8]:
tokenized_train = dataset_train.map(tokenize_function, batched=True)
tokenized_val = dataset_val.map(tokenize_function, batched=True)
tokenized_test = dataset_test.map(tokenize_function, batched=True)

tokenized_expert_45 = dataset_expert_45.map(tokenize_function, batched=True)
tokenized_expert_90 = dataset_expert_90.map(tokenize_function, batched=True)
tokenized_expert_180 = dataset_expert_180.map(tokenize_function, batched=True)

Map: 100%|██████████| 180/180 [00:00<00:00, 20371.14 examples/s]


In [9]:
columns_to_keep = ['input_ids', 'attention_mask', 'labels']

tokenized_train = tokenized_train.remove_columns(['text'])
tokenized_val = tokenized_val.remove_columns(['text'])
tokenized_test = tokenized_test.remove_columns(['text'])

tokenized_expert_45 = tokenized_expert_45.remove_columns(['text'])
tokenized_expert_90 = tokenized_expert_90.remove_columns(['text'])
tokenized_expert_180 = tokenized_expert_180.remove_columns(['text'])

## Training

In [10]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [88]:
def train_model(train_ds, eval_ds, name, epochs, batch_size, lr):
    print(f"\n{'='*50}")
    print(f"Обучение: {name}")
    print(f"{'='*50}")

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=3, problem_type="single_label_classification"
    )

    args = TrainingArguments(
        output_dir=f"./results_{name}",
        eval_strategy="epoch" if eval_ds else "no",
        save_strategy="epoch" if eval_ds else "no",
        logging_steps=50,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        learning_rate=lr,
        weight_decay=0.01,
        load_best_model_at_end=True if eval_ds else False,
        metric_for_best_model="f1_macro" if eval_ds else None,
        report_to="none",
        fp16=True if device == "cuda" else False,
        seed=42,
        disable_tqdm=False
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=eval_ds,
        compute_metrics=compute_metrics
    )

    trainer.train()
    return trainer

In [12]:
trainer_full = train_model(
    tokenized_train, tokenized_val, "full_supervised",
    epochs=3, batch_size=16, lr=2e-5
)


Обучение: full_supervised


Loading weights: 100%|██████████| 55/55 [00:00<00:00, 1202.08it/s, Materializing param=bert.pooler.dense.weight]                              
BertForSequenceClassification LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/archi

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted
1,0.598265,0.567563,0.750333,0.752370,0.752370
2,0.562799,0.554075,0.756333,0.759153,0.759153
3,0.541385,0.554486,0.758333,0.760844,0.760844


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 11.00it/s]
There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.beta', 'bert.embeddings.LayerNorm.gamma', 'bert.encoder.layer.0.attention.output.LayerNorm.be

In [72]:
trainer_expert_45 = train_model(
    tokenized_expert_45, None, "expert_few_shot",
    epochs=50, batch_size=5, lr=2e-4
)


Обучение: expert_few_shot


Loading weights: 100%|██████████| 55/55 [00:00<00:00, 1506.83it/s, Materializing param=bert.pooler.dense.weight]                              
BertForSequenceClassification LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/archi

Step,Training Loss
75,0.283013
150,0.003484
225,0.001862
300,0.001329
375,0.001073
450,0.000985


In [99]:
trainer_expert_90 = train_model(
    tokenized_expert_90, None, "expert_few_shot",
    epochs=50, batch_size=10, lr=1e-4
)


Обучение: expert_few_shot


Loading weights: 100%|██████████| 55/55 [00:00<00:00, 1148.12it/s, Materializing param=bert.pooler.dense.weight]                              
BertForSequenceClassification LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/archi

Step,Training Loss
50,0.677793
100,0.052198
150,0.010306
200,0.006278
250,0.004642
300,0.003728
350,0.003294
400,0.002976
450,0.002870


In [100]:
trainer_expert_180 = train_model(
    tokenized_expert_180, None, "expert_few_shot",
    epochs=38, batch_size=15, lr=2e-5
)


Обучение: expert_few_shot


Loading weights: 100%|██████████| 55/55 [00:00<00:00, 1238.17it/s, Materializing param=bert.pooler.dense.weight]                              
BertForSequenceClassification LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/archi

Step,Training Loss
50,1.028611
100,0.819336
150,0.570818
200,0.399399
250,0.283830
300,0.200002
350,0.158592
400,0.136857
450,0.122516


## Baseline

In [105]:
from torch.utils.data import DataLoader

model_zeroshot = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=3, 
    problem_type="single_label_classification"
).to(device)

model_zeroshot.eval()

test_dataset = tokenized_test.with_format("torch", columns=["input_ids", "attention_mask", "labels"])
dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False)

training_args = TrainingArguments(
    output_dir="./temp_zeroshot",
    per_device_eval_batch_size=32,
    report_to="none",
    disable_tqdm=True
)

trainer_zeroshot = Trainer(
    model=model_zeroshot,
    args=training_args,
    eval_dataset=test_dataset,
    compute_metrics=lambda eval_pred: {
        "accuracy": __import__('sklearn.metrics').metrics.accuracy_score(eval_pred.label_ids, eval_pred.predictions.argmax(-1)),
        "f1_macro": __import__('sklearn.metrics').metrics.f1_score(eval_pred.label_ids, eval_pred.predictions.argmax(-1), average='macro'),
        "f1_weighted": __import__('sklearn.metrics').metrics.f1_score(eval_pred.label_ids, eval_pred.predictions.argmax(-1), average='weighted')
    }
)

Loading weights: 100%|██████████| 55/55 [00:00<00:00, 1391.01it/s, Materializing param=bert.pooler.dense.weight]                              
BertForSequenceClassification LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/archi

In [106]:
res_zeroshot = evaluate_trainer(trainer_zeroshot, test_dataset, "Zero-Shot (Base Model)")

{'eval_loss': '1.094', 'eval_model_preparation_time': '0.001', 'eval_accuracy': '0.3797', 'eval_f1_macro': '0.358', 'eval_f1_weighted': '0.358', 'eval_runtime': '2.104', 'eval_samples_per_second': '7129', 'eval_steps_per_second': '222.9'}

Результаты модели: Zero-Shot (Base Model)
   Accuracy:     0.3797
   F1 Macro:     0.3580
   F1 Weighted:  0.3580


## Evaluation

In [16]:
res_full = evaluate_trainer(trainer_full, tokenized_test, "Full Supervised (~42k примеров)")


Результаты модели: Full Supervised (~42k примеров)
   Accuracy:     0.7562
   F1 Macro:     0.7585
   F1 Weighted:  0.7585


In [76]:
res_expert_45 = evaluate_trainer(trainer_expert_45, tokenized_test, "Few-Shot (45 примеров)")


Результаты модели: Few-Shot (45 примеров)
   Accuracy:     0.5767
   F1 Macro:     0.5781
   F1 Weighted:  0.5781


In [101]:
res_expert_90 = evaluate_trainer(trainer_expert_90, tokenized_test, "Few-Shot (90 примеров)")


Результаты модели: Few-Shot (90 примеров)
   Accuracy:     0.5435
   F1 Macro:     0.5499
   F1 Weighted:  0.5499


In [102]:
res_expert_180 = evaluate_trainer(trainer_expert_180, tokenized_test, "Few-Shot (180 примеров)")


Результаты модели: Few-Shot (180 примеров)
   Accuracy:     0.6016
   F1 Macro:     0.5995
   F1 Weighted:  0.5995
